- Convert the data to lower case for training and testing (Note that this implies that you also need to convert all user input to lower case, both when manually entering test cases in this part of the assignment and, importantly, during user input for the dialog system in Part 1b.)
- Split the full dataset into a training part of 85% and a test part of 15%. 

##### Splitting strategy
You may notice that many utterances in the dialogs are not unique, i.e., the exact same sentence is spoken by users in different dialogs. This influences machine learning, because even with a train-test split the same sentence may appear in both the train and test set, leading to data leakage. Therefore, for each ML model, you must train and evaluate two variants:

1. Original split. 85/15 random split of the full dataset.
2. Grouped split. Perform an 85/15 split on the full dataset, but make sure that all duplicates are in the same split. This guarantees no identical utterances leak across train and test.

Use stratified sampling for both splits, so that the class distribution in the test set matches the class distribution in the training set. This is important because rare classes might otherwise disappear entirely from the test set.

Discuss the differences in performance between each pair of variant models in the report.

In [16]:
### Import and manage data 

data_raw = open("data/dailog_acts.dat")

dialog_acts = []
utterances = []

for line in data_raw:
    line = line.strip().lower()    # /n weghalen en lowercase

    dialog_act, utterance = line.split(" ", 1)
    
    dialog_acts.append(dialog_act)
    utterances.append(utterance)      
    
    #print(line)
                      
    
### Create splits
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit # Maybe this one?

# 1. Original split
X_train, X_test, y_train, y_test = train_test_split(utterances, dialog_acts, test_size=0.15, stratify=dialog_acts)

# 2. TODO: Grouped split


### Classifiers

You will implement, train, evaluate, and compare a range of dialog act classifiers. Always think about the data you’re working with to develop your models (i.e. think about making sure you’re not (accidentally) ‘training’ on your test data). 

1. Implement a rule-based baseline based on keyword matching. An example rule could be: anytime an utterance contains ‘goodbye’, it would be classified with the dialog act bye. A reasonable performance for this baseline is at least an accuracy of 0.80. Note that the rule-based baseline must be manually constructed by reading the ‘train‘ sentences yourself and thinking about suitable keywords. It is not the goal of the baseline to run statistics on the sentences automatically and choose keywords based on that.

2. Train two ML classifiers on a bag-of-words representation of the utterances. Possible classifiers include (but are not limited to) logistic regression, SVM, MLP (feed-forward neural net), or decision trees. Motivate your choices in the report. Don’t forget to handle out-of-vocabulary (OOV) tokens explicitly, by for example dropping them, but be aware of the behavior and discuss it.

3. Train the same two ML classifiers on top of frozen pretrained embeddings. You can use DistilBERT (https://huggingface.co/distilbert/distilbert-base-uncased), which is a lightweight pretrained transformer model. Encode each utterance into a single fixed-size vector and use those vectors as features for your classifiers (logistic regression, SVM, MLP, etc.). This design keeps the pretrained model as a fixed feature extractor and lets you compare directly against your bag-of-words versions: same classifier, different representation.

4. [Optional]: Finetune the same pretrained encoder end-to-end with a classification head instead of using frozen embeddings. This means updating the transformer weights during training, not just the classifier on top. If you take this route:
    1. Compare results against the frozen-embedding version.
    2. Finetuning can be done on Google Colab's free GPU or locally on a MacBook via MPS (feasible for DistilBERT-scale models). Finetuning on CPU is not recommended.
    3. Report training time, model size, and whether the accuracy gain justifies the added compute.
    

In [17]:
### 1. Rule-based baseline

keyword = { 
    "ack"     : ["okay"],
    "affirm"  : ["yes", "right"],
    "bye"     : ["see you", "bye", "goodbye"],
    "confirm" : ["is it"],
    "deny"    : ["dont", "not"],
    "hello"   : ["hi", "hello"],
    "inform"  : ["looking for"],
    "negate"  : ["no", "not"],
    "null"    : ["cough"],
    "repeat"  : ["repeat"],
    "reqalts" : ["how about"],
    "reqmore" : ["more"],
    "request" : ["what"],
    "restart" : ["restart", "start over"],
    "thankyou": ["thank", "thanks"]
}


def rule_based_baseline(utterance):
    
    for dialog_act, keywords in keyword.items():
        for word in keywords:
            if word in utterance:
                return dialog_act


            
### 2. Train two ML classifiers on a bag-of-words representation of the utterances.

# Create bag-of-words representation
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow  = vectorizer.transform(X_test)

# 1. Logistic regression
from sklearn.linear_model import LogisticRegression

log_reg_classifier = LogisticRegression()
log_reg_classifier.fit(X_train_bow, y_train)

y_predictions = log_reg_classifier.predict(X_test_bow)


# 2. Todo...



### 3. Train the same two ML classifiers on top of frozen pretrained embeddings.

# 1. Logistic regression


# 2. Todo...



### Evaluation
Carry out an evaluation of your systems. At minimum, cover the following (in addition to anything mentioned above):

- Quantitative evaluation: Report accuracy and balanced accuracy, plus any additional metrics you think is informative. Motivate your choice.

- Error analysis: Are there specific dialog acts that are more difficult to classify? Are there particular utterances that are hard to classify (for all systems)? And why? Note: this analysis is about real utterances from the dataset. 

- Difficult cases: Come up with two types of ‘difficult instances’, for example utterances that are not fluent (e.g. due to speech recognition issues) or the presence of negation (I don’t want an expensive restaurant). For each case, create test instances and evaluate how your systems perform on these cases. Note: this analysis is about sentences that you write yourself with the goal to have the system produce an incorrect result.

- Model and split comparison: How do the models compare against each other? What is the influence of data leakage? Which one would you choose for your dialog system?

In [18]:
### Quantitive ecaluation   (HIER 1 FUNCTIE VAN MAKEN!!!!!!!!!)

from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score

def evaluate(y_test, y_pred):
    accuracy = accuracy_score(y_test, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
    
    print("Accuracy:", accuracy)
    print("Balanced accuracy", balanced_accuracy)
    
    
evaluate(y_test, y_predictions)

Accuracy: 0.9805555555555555
Balanced accuracy 0.7233581110230702
